In [0]:
%run  /Workspace/Users/rana@ghazzi.com/Dev/wikimedia/resources/notebooks/params_config

In [0]:
%sql
DESCRIBE TABLE workspace.bronze.wikimedia_bronze;
DESCRIBE TABLE workspace.silver.wikimedia_silver;
--and bot = false

In [0]:
%sql
SELECT wiki, title, COUNT(*) AS cnt
FROM workspace.silver.wikimedia_silver
GROUP BY wiki, title
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT s.wiki, s.title, s.timestamp AS silver_ts, b.max_bronze_ts
FROM workspace.silver.wikimedia_silver s
JOIN (
  SELECT wiki, title, CAST(MAX(timestamp) AS timestamp) AS max_bronze_ts
  FROM workspace.bronze.wikimedia_bronze
  WHERE bot = false AND type IN ('edit','new')
  AND type is not null
  GROUP BY wiki, title
) b
ON s.wiki = b.wiki AND s.title = b.title
WHERE s.timestamp < b.max_bronze_ts

In [0]:
%sql

-- silver_row_count should equal distinct_keys_bronze (one row per key, never more).
SELECT
  (SELECT COUNT(DISTINCT wiki, title) FROM workspace.bronze.wikimedia_bronze WHERE bot = false AND type IN ('edit','new') AND type is not null) AS distinct_keys_bronze,
  (SELECT COUNT(*) FROM workspace.silver.wikimedia_silver) AS silver_row_count

In [0]:
%sql

 DESCRIBE HISTORY workspace.silver.wikimedia_silver

In [0]:
%sql
SELECT COUNT(*) FROM workspace.silver.wikimedia_silver
WHERE id IS NULL OR wiki IS NULL OR title IS NULL OR timestamp IS NULL

In [0]:
%sql
ALTER TABLE workspace.silver.wikimedia_silver SET TBLPROPERTIES (delta.enableRowTracking=true)

In [0]:
%sql

SELECT * FROM table_changes('workspace.silver.wikimedia_silver', 19)
ORDER BY _commit_version, wiki, title

In [0]:
%sql
SELECT * FROM table_changes('workspace.silver.wikimedia_silver', 19)
WHERE _change_type = 'delete'

In [0]:
%sql
select*  FROM workspace.silver.wikimedia_silver 
--WHERE id IS NULL OR wiki IS NULL OR title IS NULL OR timestamp IS NULL

In [0]:
%sql
describe history workspace.bronze.wikimedia_bronze

In [0]:
%sql
-- Option 1: See what the table looked like at version 5 (full snapshot via time travel)
SELECT * FROM workspace.silver.wikimedia_silver@v5;

-- Option 2: See only the rows that changed in version 5 (requires CDF / row tracking enabled)
SELECT * FROM table_changes('workspace.silver.wikimedia_silver', 5, 5)
ORDER BY wiki, title;

-- Option 3: Inspect the commit metadata for version 5
DESCRIBE HISTORY workspace.silver.wikimedia_silver 

In [0]:
from pyspark.sql.functions import current_timestamp, col, get_json_object
from pyspark.sql import functions as F
read = (
    spark.table("workspace.bronze.wikimedia_bronze")
        .filter((col("bot") == False) & (col("type").isin("edit", "new")))
        .withColumn("timestamp", col("timestamp").cast("timestamp"))
        .filter(col("id").isNotNull())
        .select("id", "title", "comment", "title_url", "timestamp", "type", "user", "wiki")
)


In [0]:
%sql
SELECT * FROM workspace.silver.wikimedia_silver;

In [0]:
%sql
-- Option 1: See what the table looked like at version 5 (full snapshot via time travel)
SELECT * FROM workspace.silver.wikimedia_silver@v12;


In [0]:
%sql
SELECT * FROM workspace.silver.wikimedia_silver@v12
EXCEPT
SELECT * FROM workspace.silver.wikimedia_silver@v11;

Silver Delta table — if the bad merge already wrote incorrect data, Delta Lake supports time travel: RESTORE TABLE {catalog}.{silver_schema}.{silver_table} TO VERSION AS OF <n> (or TIMESTAMP AS OF

In [0]:
%sql
RESTORE TABLE workspace.silver.wikimedia_silver TO VERSION AS OF 12
